# CALCUALTE DIAGNOSIS AGE 

The script calculates both the diagnosis age for all individuals in the AoU. The duration in days is then converted to years by dividing by 365.25. 

In [ ]:
# Import the packages
import os
import pandas as pd
from datetime import datetime
from google.cloud import bigquery
from functools import reduce

# Retrieve the value of the environment variable
my_bucket = os.getenv("WORKSPACE_BUCKET")
# Display the value of 'my_bucket'
my_bucket


In [ ]:
# =====================================================================================
# Calculate age at aml diagnosis 
# =====================================================================================

time_to_event= f'{my_bucket}/data/covariates/aml_phenotype_time_to_event.tsv'
ehr_time = pd.read_csv(time_to_event, sep='\t', dtype={2: str})

ehr_time = ehr_time[~(ehr_time['concept_id'].isna())]
ehr_time = ehr_time[['person_id','entry_datetime']]

observation_age_period= f'{my_bucket}/data/covariates/observation_age_period.tsv'
observation = pd.read_csv(observation_age_period, sep='\t')

age_event = (
    ehr_time.merge(observation, 
               on ="person_id", 
               how ="inner")
)

age_event["date_of_birth"] = (
    pd.to_datetime(age_event["date_of_birth"]).dt.tz_localize(None)
)

age_event['entry_datetime'] = pd.to_datetime(age_event['entry_datetime'], errors='coerce').dt.tz_localize(None)
age_event['date_of_birth'] = pd.to_datetime(age_event['date_of_birth'], errors='coerce').dt.tz_localize(None)

age_event['age_at_dx'] = (
    (age_event['entry_datetime'] - age_event['date_of_birth']).dt.days / 365.25).fillna(age_event['age']
)
    
age_event.to_csv(f"{my_bucket}/data/covariates/diagnosis_age.tsv", 
                    sep ='\t', 
                    index=False)   